# 05: 多分辨率 Leiden 聚类

以多个 Leiden 分辨率运行聚类并对结果进行可视化比较，帮助选择最符合生物学粒度的分群方案。

**上游**：04 产出的嵌入结果
**本 notebook 产出**：
- `obs["leiden_res_{resolution}"]` 列（各分辨率聚类标签）
- 各分辨率着色的 UMAP 图
- 聚类指标对比表与群大小分布图
- 待研究者审查的 Stage 05 draft checkpoint（本 PR 不自动提升）

**扩展槽**（注释掉的 cell）：任何写出 `obs["{method}_clusters"]` 的聚类方法
均可与 `leiden_res_*` 列共存。ACDC 不是默认依赖，安装后取消注释即可插入。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：04（多方法嵌入），只读 verified promoted run
- **下游**：06（多方法注释），只能消费后续已提升的 05 run

### 为什么要迭代回跑？
聚类分群的分辨率选择直接影响下游注释的质量。如果在 06（注释时发现
某个细胞类型被错误拆分或合并、LLM 判决分歧高）发现问题，可能需要：
- 调整 `RESOLUTIONS` 列表（加更细或更粗的分辨率）
- 换用不同的嵌入（改 `USE_REP` 指向 04 的另一个嵌入）
- 换用 04 的另一个版本（不同嵌入方法/参数组合）

### 如何回跑（三步操作）
1. **改 `UPSTREAM_RUN_ID`**——选择已提升的 Stage 04 run
2. **改 `RUN_ID`**——每次调参使用新 run ID，不覆盖旧运行
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `RESOLUTIONS` 或 `USE_REP`）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`RUN_ID`**：每次调参重跑使用新值，旧 run 不覆盖。
- **`draft`**：本 notebook 产出的候选聚类，固定为 `NEEDS_REVIEW`。
- **`promoted`**：后续 PR5 接入研究者选择与审计后才能提升。
- **下游取数**：06 只能读取已提升 Stage 05，本 notebook 的 draft 不可直接消费。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"05_clustered"`（本 stage 标识）
- `status` = `"NEEDS_REVIEW"`（未选择分群列，禁止提升）
- `upstream` = 本次读入的上游文件路径列表
- `run_id` = 本次调参运行的唯一标识

如需查询"05 有哪些 run？各自依赖哪个 04 run？"，
查看 `results/runs/<RUN_ID>/draft/manifest.json` 中的 inputs 与 hash 即可。

In [ ]:
# === PARAMS ===

UPSTREAM_RUN_ROOT = "results/runs"
UPSTREAM_RUN_ID = "04-selected-run"
RUN_ROOT = "results/runs"
RUN_ID = "05-clustering-run"
OUTPUT_FILENAME = "05_clustered.h5ad"

# --- 使用的嵌入 ---
USE_REP = "X_pca_harmony"

# --- 邻居图（支持 Scalar-or-Sweep）---
RECOMPUTE_NEIGHBORS = False         # True = 用下方参数重新计算（与 04 的 k 不同时用）
N_NEIGHBORS = 15                    # 单值 | 列表如 [10, 15, 20, 30] → 对比不同 k 对分群的影响
N_PCS_USE_FOR_NEIGHBORS = None      # None = 使用嵌入全部维度（适合 Harmony/scVI 已降维的输出）
                                    # 设整数（如 30）= 对 X_pca 等高维嵌入截断前 N 维
MIN_CLUSTER_SIZE = 20               # 小于此值的 cluster 统计功效弱，标记 ⚠️

# --- Leiden 分辨率（天然列表 = sweep）---
# 生物学锚点参考：
#   - 胃黏膜 ~5 大类（上皮/免疫/基质/内皮/神经）→ cluster 数 10-15 对应 res ≈ 0.4-0.8
#   - 发现新亚型 → 故意过聚类 res ≥ 1.5，再按 dendrogram 合并
#   - 已知高异质性组织 → 宽范围 sweep [0.2-2.0]
RESOLUTIONS = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]

# --- 聚类稳定性分析（默认关闭——开销大，需 PI 自觉开启）---
STABILITY_ENABLED = False           # 稳定性评估需多次子抽样重算 Leiden，计算开销随 RESOLUTIONS 长度线性增长
STABILITY_N_ITER  = 10              # 子抽样次数
STABILITY_SUBSAMPLE_FRAC = 0.8      # 每次抽 80%

# --- 分辨率选择建议：预期 cluster 数范围（按组织类型调整）---
EXPECTED_CLUSTER_MIN = 5            # 胃黏膜 ~5 大类（上皮/免疫/间质/内分泌/增殖）
EXPECTED_CLUSTER_MAX = 20           # 含主要亚型

# --- UX-2 跨数据集来源构成检测（判断 cluster 是否被单一来源主导）---
# 为什么做：某 cluster 若绝大多数细胞来自单一数据集/样本，既可能是真实的数据集特异细胞群，
# 也可能是 04 批次校正不完全的残留。二者处理相反（前者保留、后者回 04 重校正），需回 04
# UMAP 的 batch 着色图人工判断。本 stage 只负责标记出可疑 cluster，不下结论。
BATCH_DOMINANCE_THRESHOLD = 0.8     # 单一来源占比超过此值即标记为疑似 batch-driven
COMPOSITION_KEYS = ["source_dataset", "sample_id", "donor_id", "disease"]
# 来源构成分层维度（运行时只取 obs 中真实存在的列）：
#   source_dataset=数据集来源  sample_id=样本  donor_id=供体  disease=疾病

# --- Consensus Clustering ---
CONSENSUS_CLUSTERING = False        # True=在多个分辨率上聚类取 consensus（计算密集，可选）

# --- Marker 基因预览（每个分辨率的 quick sanity check）---
MARKER_PREVIEW_ENABLED = True
MARKER_PREVIEW_N_GENES = 3          # 每个 cluster 显示 top N 基因

# --- 过聚类策略 ---
OVER_CLUSTERING_RES = None          # 非 None 时标记该分辨率为"故意过聚类"
# --- 锚定 marker：验证基本分离格局是否正确（不是正式注释，只是 sanity check）---
SANITY_MARKERS = {
    "Epithelial": "EPCAM",
    "Immune": "PTPRC",
    "Stromal": "VIM",
    "Proliferating": "MKI67",
}

# 胃粘膜特异性标记基因（补充 SANITY_MARKERS 之外的精细谱系）
# 为什么需要额外标记？SANITY_MARKERS 只能区分上皮/免疫/间质/增殖四大 compartment，
# 无法在聚类阶段区分胃粘膜各上皮亚型（壁细胞、主细胞、黏液细胞、内分泌细胞、肠化等）。
# 这些标记在 UMAP 上可视化后，PI 可快速判断聚类是否捕获了预期的胃上皮谱系。
GASTRIC_MARKERS = {
    "Parietal": "ATP4A",         # 壁细胞（质子泵 α 亚基）
    "Chief": "PGA3",             # 主细胞（胃蛋白酶原 A3）
    "Mucous_surface": "MUC5AC",  # 表面黏液细胞
    "Mucous_neck": "MUC6",       # 颈黏液细胞
    "Enteroendocrine": "CHGA",   # 内分泌细胞（嗜铬粒蛋白 A）
    "Pit_cell": "TFF1",          # 胃小凹细胞（三叶因子 1）
    "IM_marker": "CDX2",         # 肠化标志（尾型同源盒转录因子 2）
}

RANDOM_SEED    = 42

# --- 决策参数：研究者显式选择聚类分辨率（系统不自动选）---
# 含义：最终用于下游 06 注释的聚类列名，形如 "leiden_res_0.6"。
# 默认 None：本 stage 停在 NEEDS_REVIEW，等待研究者对照下方 UMAP / 群大小 / 来源构成 / 风险表
#           综合判断后，在此显式填入选定的 leiden_res_{r} 列名再重跑。
# 何时改：看完下方所有诊断图表、确定某一分辨率最符合生物学预期后再改。
# 改的风险：选得过高会过度碎片化（人为亚群），过低会掩盖真实亚型；此决策直接决定 06 注释粒度，
#          必须由研究者依据生物学先验判断，不能交给指标自动选。
SELECTED_CLUSTER_KEY = None
SELECTION_RATIONALE = ""            # 一句话选择依据，写入 run manifest 供追溯

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_05", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# 环境自检（每次运行自动检测平台 + conda 环境 + 关键包，见 ADR-0012）
try:
    from scrna_integration.platform import env_check
    env_check()
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

In [ ]:
# 导入（scanpy 原生 API + 框架函数仅在真正有缺口时使用）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import json
import re
import warnings
from pathlib import Path
from scipy.stats import median_abs_deviation
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    resume_run, sha256_file, snapshot_effective_parameters, validate_checkpoint,
)

# 聚类指标计算已内联至本 notebook cell（不再从 scorers import，符合 src/notebook 边界铁律）

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sc.settings.figdir = "results/figures"

## 加载上游数据

读取 04 产出的嵌入结果，确认数据维度与嵌入键名。
后续所有操作（邻居图、聚类、UMAP）均基于此 AnnData 对象。

In [ ]:
# === 只读取 verified promoted Stage 04 上游 ===
upstream_run = resume_run(UPSTREAM_RUN_ROOT, UPSTREAM_RUN_ID, promoted=True)
upstream_manifest_path = upstream_run.promoted_dir / "manifest.json"
upstream_manifest = json.loads(upstream_manifest_path.read_text(encoding="utf-8"))
if upstream_manifest.get("run_id") != UPSTREAM_RUN_ID:
    raise ValueError("upstream manifest run_id 与配置不匹配")
if upstream_manifest.get("stage") != "04_embedded":
    raise ValueError("upstream manifest stage 必须是 04_embedded")
upstream_stage_status = upstream_manifest.get("stage_status")
if upstream_stage_status == "SUCCESS_WITH_WARNINGS":
    warning_acceptance = upstream_manifest.get("warning_acceptance")
    if not isinstance(warning_acceptance, dict) or not all(
        isinstance(warning_acceptance.get(field), str) and warning_acceptance[field].strip()
        for field in ("accepted_by", "accepted_at")
    ):
        raise ValueError("SUCCESS_WITH_WARNINGS 上游缺少有效 warning_acceptance")
elif upstream_stage_status != "SUCCESS":
    raise ValueError("upstream manifest stage_status 不可供 Stage 05 消费")
UPSTREAM_CHECKPOINT = validate_checkpoint(upstream_manifest_path)
upstream_input = {
    "run_id": UPSTREAM_RUN_ID, "stage": "04_embedded",
    "manifest_path": str(upstream_manifest_path),
    "manifest_sha256": sha256_file(upstream_manifest_path),
    "checkpoint_path": str(UPSTREAM_CHECKPOINT),
    "checkpoint_sha256": sha256_file(UPSTREAM_CHECKPOINT),
}
print("加载上游:", UPSTREAM_CHECKPOINT)
adata = sc.read_h5ad(UPSTREAM_CHECKPOINT)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"use_rep='{USE_REP}' -- 存在: {USE_REP in adata.obsm}")
# === promoted Stage 04 上游读取完成 ===

In [ ]:
# === expression_contract 校验（05 只读 X——X 必须为 normalized_log1p，stage=03）===
try:
    from scrna_integration.run_contract import validate_expression_contract
except ImportError:
    pass  # P0-a 未合并时优雅降级；当前 main 已含，本分支实际会成功
else:
    _contract = validate_expression_contract(adata, expected_scale="normalized_log1p", stage="03")
    print(f"expression_contract 验证通过: x_scale={_contract['x_scale']}, "
          f"counts_layer={_contract['counts_layer']}, stage={_contract['stage']}")

# === USE_REP 维度匹配检查 ===
# 邻居图构建前检查嵌入维度，帮助判断是否需要截断。
# Harmony/scVI 输出已是低维 → 使用全部维度。
# X_pca 可能是高维 → 建议用 N_PCS_USE_FOR_NEIGHBORS 截断前 N 维。
_embed_dim = adata.obsm[USE_REP].shape[1]
print(f"USE_REP = '{USE_REP}', 维度 = {_embed_dim}")

_n_pcs_neighbors = N_PCS_USE_FOR_NEIGHBORS  # None 或整数

if USE_REP in ("X_pca",) and _embed_dim > 30 and _n_pcs_neighbors is None:
    print(f"  ⚠️ 使用原始 PCA 全维度({_embed_dim}) 构建邻居图——包含噪音维度")
    print(f"  → 建议设 N_PCS_USE_FOR_NEIGHBORS = 30 或改用 X_pca_harmony/X_scVI")
elif _n_pcs_neighbors is not None:
    print(f"  邻居图使用前 {_n_pcs_neighbors} 维（N_PCS_USE_FOR_NEIGHBORS）")
else:
    print(f"  ✓ 嵌入已是低维({_embed_dim}D)，使用全部维度")

## 计算邻居图

在 04 选定嵌入（`USE_REP`）上构建 k 近邻图。
**为什么所有分辨率共享同一个邻居图？**
不同分辨率只改变聚类的粒度（通过 Leiden 算法的 resolution 参数），
不需要每次重新计算细胞间距离关系。共享避免了重复计算和内存浪费。

`n_pcs=None`：嵌入已是低维表示，无需再做 PCA 降维。

### N_NEIGHBORS 如何影响聚类

邻居图是 Leiden 聚类的**唯一输入**——resolution 只改变"切多细"，但哪些细胞是"邻居"由 k 决定。

| k 值 | 效果 | 适用场景 |
|---|---|---|
| 10 | 局部敏感，稀有群体可见，但噪音连接多 | 找稀有细胞类型（<1%）、transitional states |
| 15 | 平衡（默认） | 大多数分析（50k-200k cells） |
| 20-30 | 全局稳定，减少碎片化 | 数据噪音大、batch effect 残留明显 |
| 50+ | 非常粗粒度 | >200k cells 或只需大类注释 |

**经验法则**：`k ≈ sqrt(n_cells) / 3`，但最终以生物学合理性为准。

**怎么判断 k 是否合适**：
- res=0.8 只分出 3-4 个大群 → 试**减小 k**（让局部结构可见）
- res=0.8 产出 20+ 个碎片 → 试**增大 k**（让全局结构主导）
- 同一 compartment 内的亚型总是被合并 → k 可能太大

**与 resolution 的交互**：k 决定"谁和谁是邻居"，resolution 决定"切多细"。先确定合适的 k，再 sweep resolution。如果 k 不对，sweep resolution 无法补救。

In [ ]:
# === USE_REP 存在性守卫 ===
_available_reps = [k for k in adata.obsm.keys() if k.startswith("X_")]
print(f"可用嵌入 (obsm): {_available_reps}")

if USE_REP not in adata.obsm:
    print(f"⚠️ USE_REP='{USE_REP}' 不在 obsm 中！")
    print(f"  可用选项: {_available_reps}")
    if "X_pca_harmony" in adata.obsm:
        USE_REP = "X_pca_harmony"
        print(f"  → 自动回退到: {USE_REP}")
    elif "X_pca" in adata.obsm:
        USE_REP = "X_pca"
        print(f"  → 自动回退到: {USE_REP}")
    else:
        raise KeyError(f"无可用嵌入！检查上游 04_embedded 是否正确运行。obsm keys: {_available_reps}")
else:
    print(f"✓ USE_REP='{USE_REP}' 确认存在")

# 在选定嵌入上计算邻居图。
# 支持 Scalar-or-Sweep：列表模式自动对比不同 k 对分群的影响。
# 单值模式：直接计算，所有分辨率共享同一邻居图。
# _n_pcs_neighbors 来自上游维度检查 cell，None=使用全部维度。
_k_values = N_NEIGHBORS if isinstance(N_NEIGHBORS, list) else [N_NEIGHBORS]

if RECOMPUTE_NEIGHBORS or len(_k_values) > 1:
    if len(_k_values) > 1:
        # Sweep 模式：对每个 k 计算 Leiden(res=1.0)，对比 cluster 数差异
        print(f"N_NEIGHBORS sweep: {_k_values}")
        k_comparison = []
        for k in _k_values:
            sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                            n_neighbors=k, random_state=RANDOM_SEED)
            sc.tl.leiden(adata, resolution=1.0, key_added=f"leiden_k{k}_res1.0",
                         flavor="igraph", random_state=RANDOM_SEED)
            n_clusters = adata.obs[f"leiden_k{k}_res1.0"].nunique()
            k_comparison.append({"k": k, "n_clusters_at_res1.0": n_clusters})
        print(pd.DataFrame(k_comparison).to_string(index=False))
        # 最终使用列表最后一个 k
        sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                        n_neighbors=_k_values[-1], random_state=RANDOM_SEED)
    else:
        print(f"RECOMPUTE_NEIGHBORS=True，重新计算 k={_k_values[0]}")
        sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                        n_neighbors=_k_values[0], random_state=RANDOM_SEED)
else:
    print(f"\n===== Neighbors on {USE_REP} =====")
    sc.pp.neighbors(adata, use_rep=USE_REP, n_pcs=_n_pcs_neighbors,
                    n_neighbors=_k_values[-1], random_state=RANDOM_SEED)

print(f"Neighbor graph: {adata.obsp['connectivities'].shape}")
print(f"  n_neighbors={adata.uns['neighbors']['params']['n_neighbors']}")

## 多分辨率 Leiden 聚类

每个分辨率各做一次 Leiden 聚类。所有 `leiden_res_*` 列共存以供对比。
使用 `flavor="igraph"` 以兼容 scanpy >= 1.10。

**为什么做多个分辨率？** 没有"正确"的分辨率——不同生物学问题需要
不同粒度。粗分辨率（0.2-0.4）捕获大细胞谱系；细分辨率（1.0+）
区分亚型。PI 查看 UMAP 后根据生物学背景选择最合理的一个或几个。

In [ ]:
# 多分辨率 Leiden 聚类：每个分辨率一个 obs 列。
print(f"\n===== Leiden: {len(RESOLUTIONS)} resolutions =====")

for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    print(f"  resolution={res} -> obs['{key}']")
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )
    n_clusters = adata.obs[key].nunique()
    print(f"    clusters: {n_clusters}" + (" ← 接近预期大类数" if 8 <= n_clusters <= 20 else ""))

leiden_columns = [c for c in adata.obs.columns if c.startswith("leiden_res_")]
print(f"\nLeiden columns produced: {leiden_columns}")

# 最小 cluster 大小警告
for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    sizes = adata.obs[key].value_counts()
    small = sizes[sizes < MIN_CLUSTER_SIZE]
    if len(small) > 0:
        print(f"  ⚠️ res={res}: {len(small)} 个 cluster < {MIN_CLUSTER_SIZE} 细胞: {dict(small)}")

In [ ]:
_stability_completed = False

# --- 聚类稳定性分析 ---
# 子抽样 ARI：对每个分辨率，随机抽取 80% 细胞重复聚类，
# 计算子集聚类与全集聚类在子集上的 ARI。
# ARI 越高 = 分群对数据扰动越鲁棒。
#
# 注意：稳定性评估对每个分辨率多次子抽样并重算 Leiden，
# 计算开销随 RESOLUTIONS 长度和 STABILITY_N_ITER 线性增长，
# 大数据集（>100k cells）慎开。建议仅在接近最终分辨率时重新开启。
if STABILITY_ENABLED:
    print("\n===== 聚类稳定性分析 =====")
    print(f"参数: n_iter={STABILITY_N_ITER}, subsample_frac={STABILITY_SUBSAMPLE_FRAC}")

    _use_k = _k_values[-1] if isinstance(N_NEIGHBORS, list) else N_NEIGHBORS
    stability_results = []

    for res in RESOLUTIONS:
        aris = []
        full_key = f"leiden_res_{res}"
        for i in range(STABILITY_N_ITER):
            np.random.seed(RANDOM_SEED + i)
            idx = np.random.choice(
                adata.n_obs,
                int(adata.n_obs * STABILITY_SUBSAMPLE_FRAC),
                replace=False,
            )
            adata_sub = adata[idx].copy()
            # 子集需要独立的 neighbors + leiden
            sc.pp.neighbors(adata_sub, use_rep=USE_REP, n_neighbors=_use_k,
                            random_state=RANDOM_SEED)
            sc.tl.leiden(adata_sub, resolution=res, key_added="leiden_sub",
                         flavor="igraph", random_state=RANDOM_SEED)
            # ARI：子集聚类标签 vs 全集聚类标签在子集上的取值
            full_labels = adata.obs[full_key].values[idx]
            ari = adjusted_rand_score(full_labels, adata_sub.obs["leiden_sub"])
            aris.append(ari)
            del adata_sub
        stability_results.append({
            "resolution": res,
            "mean_ARI": np.mean(aris),
            "std_ARI": np.std(aris),
        })

    stability_df = pd.DataFrame(stability_results)

    # 稳定性折线图
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.errorbar(stability_df["resolution"], stability_df["mean_ARI"],
                yerr=stability_df["std_ARI"], marker="o", capsize=4)
    ax.axhline(0.9, color="green", linestyle="--", alpha=0.5,
               label="ARI=0.9（高稳定性）")
    ax.axhline(0.8, color="orange", linestyle="--", alpha=0.5,
               label="ARI=0.8（可接受）")
    ax.set_xlabel("Resolution")
    ax.set_ylabel("ARI（子抽样一致性）")
    ax.set_title("聚类稳定性：越高 = 分群对数据扰动越鲁棒")
    ax.legend()
    plt.tight_layout(); plt.show()

    print(stability_df.to_string(index=False))
    _stability_completed = True
else:
    print("\n聚类稳定性分析已关闭（STABILITY_ENABLED=False）。")

In [ ]:
# --- Consensus Clustering ---
# 使用多个中间分辨率构建 co-clustering 矩阵，识别稳健的细胞对
if CONSENSUS_CLUSTERING:
    from sklearn.metrics import adjusted_rand_score

    print("===== Consensus Clustering =====")
    # 使用 3 个中间分辨率做 consensus
    _consensus_res = [r for r in RESOLUTIONS if 0.4 <= r <= 1.2][:3]
    if len(_consensus_res) >= 2:
        n = adata.n_obs
        co_cluster = np.zeros((n, n), dtype=np.float32)
        for res in _consensus_res:
            key = f"leiden_res_{res}"
            labels = adata.obs[key].values
            for cl in np.unique(labels):
                idx = np.where(labels == cl)[0]
                co_cluster[np.ix_(idx, idx)] += 1
        co_cluster /= len(_consensus_res)
        # 高置信度 pair：co_cluster == 1.0（所有分辨率都共聚）
        n_robust_pairs = int((co_cluster == 1.0).sum() - n) // 2  # 减去对角线
        n_total_pairs = n * (n - 1) // 2
        print(f"  Robust pairs（所有 {len(_consensus_res)} 分辨率共聚）: {n_robust_pairs:,} / {n_total_pairs:,} ({100*n_robust_pairs/n_total_pairs:.1f}%)")
        print(f"  ⚠️ 注意：consensus clustering 在大数据集上内存密集（n² 矩阵）")
        del co_cluster
    else:
        print("  跳过：需要至少 2 个在 [0.4, 1.2] 范围内的分辨率")


In [ ]:
# --- Marker 基因预览：每个分辨率的 top DEG（快速 sanity check）---
# 如果所有 cluster 的 top gene 都是核糖体/线粒体 → HVG 有问题，回 03 调整
if MARKER_PREVIEW_ENABLED:
    print("\n" + "=" * 60)
    print("Marker 预览：每个分辨率的 top DEG（快速 sanity check）")
    print("如果所有 cluster 的 top gene 都是核糖体/线粒体 -> HVG 有问题，回 03")
    print("=" * 60)

    for res in RESOLUTIONS:
        key = f"leiden_res_{res}"
        n_clusters = adata.obs[key].nunique()
        sc.tl.rank_genes_groups(
            adata, groupby=key, method="wilcoxon",
            n_genes=MARKER_PREVIEW_N_GENES, use_raw=False,
        )

        print(f"\n--- Resolution {res} ({n_clusters} clusters) ---")
        # 紧凑表格：每行一个 cluster，列出 top N 基因
        result = adata.uns["rank_genes_groups"]
        groups = result["names"].dtype.names
        for group in groups[:min(10, len(groups))]:  # 最多显示 10 个 cluster
            genes = [result["names"][group][i]
                     for i in range(MARKER_PREVIEW_N_GENES)]
            scores = [f"{result['scores'][group][i]:.1f}"
                      for i in range(MARKER_PREVIEW_N_GENES)]
            print(f"  Cluster {group}: {', '.join(genes)} "
                  f"(scores: {', '.join(scores)})")
        if len(groups) > 10:
            print(f"  ... 共 {len(groups)} clusters，仅显示前 10")

    # 清理——避免最后一个分辨率的 DEG 结果被误当做"最终" DEG 结果
    del adata.uns["rank_genes_groups"]
    print("\nrank_genes_groups 已清理（仅预览用，非最终 DEG 结果）。")
else:
    print("\nMarker 预览已关闭（MARKER_PREVIEW_ENABLED=False）。")

## 聚类指标对比——辅助选择最佳分辨率

对每个分辨率计算聚类质量指标，与可视化结果互为补充。

**直接遍历分辨率列表**——没有回调、没有封装。学生逐行可读。

**指标说明**（数据允许时计算）：
- **silhouette score**（轮廓系数）：衡量簇的紧密程度与分离度。值越接近 1 越好。
  在 PCA 空间上计算，反映嵌入空间中簇的结构质量。
- **ARI**（调整兰德指数）：与已知标签的一致性。需要 `obs` 中存在参考标签列。

**如何用指标辅助决策**：
- silhouette 随分辨率升高通常会缓慢下降——这是正常的（更细的簇边界更模糊）。
  重点看下降趋势中是否存在"拐点"（分辨率增加但 silhouette 骤降），而非绝对值。
- ARI 仅在存在参考标签时有意义。
- **最终决策以 UMAP 可视化为主，指标为参考**——生物学的分群合理性不能仅由数值指标决定。

对比报告写入 `results/figures/sweep_05/sweep_report.md`。

In [ ]:
_sweep_completed = False

# 显式 for 循环：遍历各分辨率，运行 Leiden 并计算聚类指标。
# 每个分辨率在独立拷贝上运行，避免列名冲突。
print("\n===== 显式遍历分辨率 + clustering_metrics =====\n")

results = []
# 单次拷贝复用：内联指标计算只读 obsm/obs，不修改 adata，
# 因此可在同一份拷贝上重跑不同 resolution 的 leiden（各分辨率写不同 obs 列），
# 避免 8 个分辨率各 deep copy 一次导致的 8x 内存峰值。
_adata_metrics = adata.copy()

for res in RESOLUTIONS:
    print(f"--- resolution={res} ---")

    key = f"leiden_res_{res}"
    sc.tl.leiden(
        _adata_metrics,
        resolution=res,
        key_added=key,
        flavor="igraph",
        random_state=RANDOM_SEED,
    )

    # === 聚类指标内联计算（原 scorers.clustering_metrics，现留 notebook cell 可见可调）===
    # 计算原理：在嵌入空间上用 silhouette_score 评估簇的紧凑度与分离度
    _m = {}
    _labels = _adata_metrics.obs[key].astype(str)
    _mask = _adata_metrics.obs[key].notna()
    _valid_labels = _labels[_mask]

    # 轮廓系数（Silhouette）：衡量簇内紧凑 vs 簇间分离，范围 [-1, 1]，越高越好
    # 在 PCA 空间上计算——反映嵌入空间中簇的结构质量
    # 注意：分辨率升高通常 silhouette 缓慢下降，重点看下降趋势中的"拐点"而非绝对值
    if USE_REP in _adata_metrics.obsm and _valid_labels.nunique() >= 2:
        _x = _adata_metrics[_mask].obsm[USE_REP]  # 使用 05 选定的嵌入空间
        try:
            _m["silhouette"] = float(
                silhouette_score(_x, _valid_labels[_mask])
            )
        except Exception:
            pass

    # ARI（调整兰德指数）：与已知标签的一致性（仅当 obs 中存在参考标签列时计算）
    # 值为 1 表示完美一致，0 表示偶然一致
    _label_key = None
    for _cand in ["cell_type", "label", "annotation"]:
        for _col in _adata_metrics.obs.columns:
            if _cand in _col:
                _label_key = _col
                break
        if _label_key:
            break
    if _label_key is not None:
        _true_labels = _adata_metrics.obs[_label_key][_mask].astype(str)
        try:
            _m["ari"] = float(
                adjusted_rand_score(_true_labels, _valid_labels[_mask])
            )
        except Exception:
            pass

    results.append({"resolution": res, **_m})

    n_cl = _adata_metrics.obs[key].nunique()
    metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in _m.items()
                            if isinstance(v, float) and not np.isnan(v))
    print(f"  簇数: {n_cl}  指标: {metrics_str}")

del _adata_metrics

# 收集为 DataFrame 对比表
sweep_df = pd.DataFrame(results)
os.makedirs("results/figures/sweep_05", exist_ok=True)

# 写 Markdown 报告
lines = ["# 05 聚类对比报告\n",
         f"**{len(RESOLUTIONS)} 个分辨率** 已评估。\n",
         "## 指标表\n"]
lines.append("| " + " | ".join(sweep_df.columns) + " |")
lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
for _, row in sweep_df.iterrows():
    vals = []
    for col in sweep_df.columns:
        v = row[col]
        if isinstance(v, float):
            vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
        else:
            vals.append(str(v))
    lines.append("| " + " | ".join(vals) + " |")
with open("results/figures/sweep_05/sweep_report.md", "w") as f:
    f.write("\n".join(lines) + "\n")

# 绘制指标随分辨率变化的折线图——辅助 PI 识别拐点
metric_cols = [c for c in sweep_df.columns
               if c != "resolution" and not sweep_df[c].isna().all()]
if metric_cols:
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(5 * len(metric_cols), 4))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        ax.plot(sweep_df["resolution"], sweep_df[col], "o-", color="#2c7bb6", markersize=6)
        ax.set_xlabel("resolution")
        ax.set_ylabel(col)
        ax.set_title(f"{col} vs resolution")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig_path = "results/figures/sweep_05/metrics_vs_resolution.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"\n指标-分辨率折线图: {fig_path}")
    plt.close("all")

# 展示对比表
print("\n聚类指标对比表:")
try:
    from IPython.display import display as ipy_display
    ipy_display(sweep_df)
except ImportError:
    print(sweep_df)

print("\n对比报告: results/figures/sweep_05/sweep_report.md")
adata.uns["05_sweep_v1"] = {
    "resolutions_swept": RESOLUTIONS,
    "scorer": "inline_silhouette_ari",
    "report_dir": "results/figures/sweep_05",
    "timestamp": datetime.datetime.now().isoformat(),
}
_sweep_completed = True


## 相邻分辨率稳定性（cluster 数变化 + ARI）

In [ ]:
_adjacent_completed = False

# 相邻分辨率稳定性（廉价指标，始终运行）：
# 复用已算好的 leiden_res 列，对相邻分辨率对 (res_i, res_{i+1}) 直接算 adjusted_rand_score，
# 无需重聚类。与 STABILITY_ENABLED 的重型子抽样（每分辨率多次重算 Leiden）互补——后者开销大
# 默认关闭，本 cell 只读已有列因而始终运行。
# 为什么看这个：ARI 骤降或 cluster 数跳变，提示该处分辨率跨越了真实结构边界（一个大群分裂为多个），
# 是选分辨率时的天然候选拐点，供研究者决策参考（不自动选）。
print("\n===== 相邻分辨率稳定性（相邻对 ARI + cluster 数变化）=====\n")

_adj = {"res_low": [], "res_high": [], "n_clusters_low": [], "n_clusters_high": [],
        "delta_clusters": [], "ari": []}
for _i in range(len(RESOLUTIONS) - 1):
    _r_lo, _r_hi = RESOLUTIONS[_i], RESOLUTIONS[_i + 1]
    _k_lo, _k_hi = f"leiden_res_{_r_lo}", f"leiden_res_{_r_hi}"
    if _k_lo not in adata.obs.columns or _k_hi not in adata.obs.columns:
        continue
    _n_lo = int(adata.obs[_k_lo].nunique())
    _n_hi = int(adata.obs[_k_hi].nunique())
    _ari = float(adjusted_rand_score(
        adata.obs[_k_lo].astype(str), adata.obs[_k_hi].astype(str)))
    _adj["res_low"].append(float(_r_lo))
    _adj["res_high"].append(float(_r_hi))
    _adj["n_clusters_low"].append(_n_lo)
    _adj["n_clusters_high"].append(_n_hi)
    _adj["delta_clusters"].append(_n_hi - _n_lo)
    _adj["ari"].append(_ari)

adjacent_stability_df = pd.DataFrame(_adj)
if len(adjacent_stability_df) > 0:
    print(adjacent_stability_df.to_string(index=False))
    # 提示 ARI 最低（结构变动最大）的相邻对——最可能是真实结构边界。
    _min_idx = adjacent_stability_df["ari"].idxmin()
    _row = adjacent_stability_df.loc[_min_idx]
    print(f"\n  变动最大的相邻对: res {_row['res_low']} → {_row['res_high']} "
          f"(ARI={_row['ari']:.3f}, cluster {int(_row['n_clusters_low'])} "
          f"→ {int(_row['n_clusters_high'])})")
    print("  → ARI 骤降处可能跨越了真实结构边界，供选分辨率时参考（不自动选）。")
else:
    print("可用相邻分辨率对不足（RESOLUTIONS 长度 < 2 或列缺失），跳过。")

# columnar dict 写回 uns（等长列表，安全 h5ad round-trip）。
adata.uns["05_stability_adjacent"] = _adj
_adjacent_completed = True

In [ ]:
# === Per-cell silhouette score（05 自身诊断用途——评估聚类边界质量）===
# 注意：此列仅作 05 内部诊断（识别边界模糊的细胞），不被 06 标注流程使用。
# 后续如 06 需要边界置信度信息，可从 05 读取此列或在 06 内独立计算。
from sklearn.metrics import silhouette_samples

# 选择一个代表性分辨率（推荐的或中间的）
_sil_res = adata.uns.get("recommended_resolution", RESOLUTIONS[len(RESOLUTIONS)//2])
_sil_key = f"leiden_res_{_sil_res}"

if _sil_key in adata.obs.columns and USE_REP in adata.obsm:
    print(f"计算 per-cell silhouette（{_sil_key}, 嵌入={USE_REP}）...")
    
    _labels = adata.obs[_sil_key].astype(str).values
    _embed = adata.obsm[USE_REP]
    
    # 对大数据集采样（>50k 太慢）
    if adata.n_obs > 50000:
        _sample_idx = np.random.choice(adata.n_obs, size=50000, replace=False)
        _sil_scores = np.full(adata.n_obs, np.nan)
        _sil_scores[_sample_idx] = silhouette_samples(_embed[_sample_idx], _labels[_sample_idx])
        print(f"  采样 50k cells 计算（总 {adata.n_obs:,}）")
    else:
        _sil_scores = silhouette_samples(_embed, _labels)
    
    adata.obs["silhouette_score"] = _sil_scores
    
    # 统计
    _mean_sil = np.nanmean(_sil_scores)
    _boundary_pct = (np.abs(_sil_scores) < 0.1).sum() / max(np.isfinite(_sil_scores).sum(), 1) * 100
    print(f"  平均 silhouette: {_mean_sil:.3f}")
    print(f"  边界细胞（|sil| < 0.1）: {_boundary_pct:.1f}%")
    print(f"  → obs['silhouette_score'] 已写入（仅供 05 自身诊断：边界模糊细胞 |sil| < 0.1 占 {_boundary_pct:.1f}%）")
else:
    print(f"⚠️ {_sil_key} 或 {USE_REP} 不可用，跳过 per-cell silhouette")

## 扩展槽——其他聚类方法

任何写出 `obs["{method}_clusters"]` 的聚类方法均可与 `leiden_res_*` 列共存
并以同样方式参与对比。

**ACDC**（已注释）：全局搜索最优分区。**不**是默认依赖（在 GCPL 数据上
运行太慢）。PI 安装后取消注释，新列自动进入 06。

**新方法的通用模式**：写 `obs["{method}_clusters"]` = labels，
然后加入显式 for 循环的 candidates 列表或单独对比——
与 04 嵌入同样的并行槽位约定，零框架改动。

In [ ]:
# # === ACDC clustering (commented out -- NOT a default dependency) ===
# # PREREQUISITE: pip install acdc_py
# # Enable by removing comments below.
#
# # import acdc_py  # 包名/导入名以 PyPI 实际为准，启用前先确认；ACDC 非默认依赖
# # # ACDC searches for an optimal partition.
# # acdc_result = ACDC.ACDC(adata, ...)
# # adata.obs["acdc_clusters"] = acdc_result.labels
# # print(f"ACDC: {adata.obs['acdc_clusters'].nunique()} clusters")
#
# print("ACDC cell is commented out. "
#       "Uncomment when acdc_py is installed and suitable for this dataset.")
#
# # === Add any future method here ===
# # Pattern: write adata.obs["{method}_clusters"] = labels
# # Then add to sweep candidates or compare standalone.
# # Example: adata.obs["foocluster_clusters"] = foo_cluster.fit_predict(
# #     adata.obsm[USE_REP])

## 各分辨率 UMAP 可视化

**按分辨率逐个着色 UMAP**——每个分辨率算完即出图。
PI 可以跑一个分辨率看一个，判断该分辨率的聚类是否合理，再决定最终采纳哪个。

**怎么看 UMAP 判断分群质量**：
- 同一簇在 UMAP 上应该聚在一起（簇内紧凑）
- 不同簇之间应该有清晰的边界或过渡
- 如果某个分辨率把一个明显的谱系拆成多块 → 过分（分辨率太高）
- 如果某个分辨率把明显不同的细胞群合并 → 欠分（分辨率太低）
- 在 UMAP 上看簇的分布，结合下方的群大小分布图，综合判断

In [ ]:
# 从已有邻居图计算 UMAP（所有分辨率共享同一嵌入）。
# 然后逐个分辨率着色，每个分辨率即时出图，方便 PI 对比选择。
print("\n===== UMAP per resolution =====")

sc.tl.umap(adata, random_state=RANDOM_SEED)

leiden_cols = sorted(
    [c for c in adata.obs.columns if c.startswith("leiden_res_")],
    key=lambda x: float(x.split("_")[-1]),
)

for col in leiden_cols:
    n_clusters = adata.obs[col].nunique()
    res_str = col.split("_")[-1]
    print(f"\n--- resolution={res_str}: {n_clusters} clusters ---")

    sc.pl.umap(
        adata, color=col,
        title=f"Leiden res={res_str}（{n_clusters} 群）",
        legend_loc="on data" if n_clusters <= 10 else "right margin",
        frameon=False,
        save=f"_05_sanity_{col}.png",
    )
    plt.close("all")

print(f"\nUMAP 出图完成，共 {len(leiden_cols)} 个分辨率。")

## 群大小分布——辅助判断分辨率合理性

除了 UMAP 可视化，各分辨率下的群大小分布也是选择分群方案的重要参考：

- **如果存在极大群和极小群并存**（一个群占了 50% 以上细胞，另一群只有几十个）→ 可能分辨率不合适
- **如果群大小相对均匀** → 分群粒度与数据结构匹配较好
- **极小的群**（< 1% 总细胞数）可能是噪声或稀有细胞亚型——需要结合生物学背景判断

下方的条形图展示每个分辨率下各群的细胞数分布。

In [ ]:
# 各分辨率下的群大小分布——帮助 PI 判断分群是否合理。
print("\n===== 群大小分布 =====")

n_res = len(RESOLUTIONS)
n_cols = 3
n_rows = (n_res + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
if n_rows == 1 and n_cols == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i, res in enumerate(RESOLUTIONS):
    key = f"leiden_res_{res}"
    cluster_sizes = adata.obs[key].value_counts().sort_index()
    n_clusters = len(cluster_sizes)

    ax = axes[i]
    colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
    ax.bar(range(n_clusters), cluster_sizes.values, color=colors)
    ax.set_title(f"resolution={res}（{n_clusters} 群）")
    ax.set_xlabel("群编号")
    ax.set_ylabel("细胞数")
    ax.set_xticks(range(n_clusters))
    ax.tick_params(axis="x", labelsize=8)

# 隐藏多余的子图
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
fig_path = "results/figures/05_cluster_sizes.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"群大小分布图保存至: {fig_path}")
plt.close("all")

## 分辨率选择建议

以下基于预期 cluster 数范围做**启发式推荐**。这不是自动选择——最终决策以 UMAP 目测为准。

**推荐逻辑**：
- 胃黏膜预期 ~5 大类（上皮/免疫/间质/内分泌/增殖），对应 5-20 个 cluster 的分辨率
- 倾向于选择该窗口内**稍高**的分辨率（宁可稍过聚类，后续通过 dendrogram 合并）
- 超过 20 个 cluster 的分辨率适合"过聚类探索"策略——配合 dendrogram 识别可合并的姐妹 cluster

**注意事项**：
- 指标仅做参考，最终决策以 UMAP 目测为准
- 关注同一簇是否紧凑、不同簇边界是否清晰、已知 marker 分布是否合理
- 如有稳定性数据，优先选择高 ARI 的分辨率

In [ ]:
# === 分辨率选择建议 ===
# 基于预期 cluster 数范围做启发式推荐。这不是自动选择——
# 最终决策以 UMAP 目测为准，指标和推荐仅做参考。
print("===== 分辨率选择建议（多指标并列——不自动替 PI 选定）=====\n")

# 1. 各分辨率的簇数与指标汇总
_n_clusters = {res: adata.obs[f"leiden_res_{res}"].nunique() for res in RESOLUTIONS}

# 收集所有可用指标的 DataFrame 并列展示
_summary_rows = []
for res in RESOLUTIONS:
    _row = {"resolution": res, "n_clusters": _n_clusters[res]}
    _summary_rows.append(_row)
_summary = pd.DataFrame(_summary_rows)

# 如果已有指标结果，并入同一张表
if _sweep_completed:
    _metrics_df = pd.DataFrame(results)
    _summary = _summary.merge(_metrics_df, on="resolution", how="left")
print(_summary.to_string(index=False))

# 2. 多模式推荐——并列展示，不自动替 PI 选定
#    每种模式给出各自的最优分辨率，PI 对照 UMAP 目测后自行决策。
print()
print("--- 多模式推荐（并列，无自动选择）---")

# 模式 1: Silhouette 最优（簇紧密+分离好）
if _sweep_completed:
    _metrics_df = pd.DataFrame(results)
    if "silhouette" in _metrics_df.columns:
        _best_sil_idx = _metrics_df["silhouette"].idxmax()
        _best_sil_res = _metrics_df.loc[_best_sil_idx, "resolution"]
        _best_sil_val = _metrics_df.loc[_best_sil_idx, "silhouette"]
        print(f"  模式1-轮廓系数最优: res={_best_sil_res} (silhouette={_best_sil_val:.4f})")
        print(f"    → 簇内紧凑+簇间分离最好，但可能漏掉稀有亚型")
    else:
        print(f"  模式1-轮廓系数: 无 silhouette 数据")

# 模式 2: 稳定性最优（子抽样 ARI 最高）
if STABILITY_ENABLED and _stability_completed:
    _best_stab_idx = stability_df["mean_ARI"].idxmax()
    _best_stab_res = stability_df.loc[_best_stab_idx, "resolution"]
    _best_stab_val = stability_df.loc[_best_stab_idx, "mean_ARI"]
    print(f"  模式2-稳定性最优: res={_best_stab_res} (mean_ARI={_best_stab_val:.4f})")
    print(f"    → 分群对数据扰动最鲁棒，适合需要可重现分群的下游分析")
    # 列出所有高稳定性候选（ARI ≥ 0.85）
    _stable_candidates = stability_df[stability_df["mean_ARI"] >= 0.85]["resolution"].tolist()
    if _stable_candidates:
        print(f"    高稳定性候选（ARI ≥ 0.85）: {_stable_candidates}")
else:
    print(f"  模式2-稳定性: 未启用（STABILITY_ENABLED=False）")

# 模式 3: 预期 cluster 数范围（生物学先验，由 PARAMS 区 EXPECTED_CLUSTER_MIN/MAX 控制）
_annot_candidates = [r for r in RESOLUTIONS
                     if EXPECTED_CLUSTER_MIN <= _n_clusters[r] <= EXPECTED_CLUSTER_MAX]
print(f"  模式3-生物学先验: 预期 {EXPECTED_CLUSTER_MIN}-{EXPECTED_CLUSTER_MAX} 簇（可在 PARAMS 区按组织类型调整）")
if _annot_candidates:
    print(f"    → 符合范围的分辨率: {_annot_candidates}")
else:
    print(f"    → 无分辨率产出 {EXPECTED_CLUSTER_MIN}-{EXPECTED_CLUSTER_MAX} 个簇")

# 模式 4: 过聚类探索（适合发现新亚型/transitional states）
_explore_candidates = [r for r in RESOLUTIONS if _n_clusters[r] > EXPECTED_CLUSTER_MAX]
print(f"  模式4-过聚类探索（> {EXPECTED_CLUSTER_MAX} 簇）: {_explore_candidates if _explore_candidates else '无'}")
if _explore_candidates:
    print(f"    → 配合 dendrogram 合并策略，适合发现新亚型/transitional states")

print(f"\n  ⚠️ 以上四种模式均为参考——不自动替 PI 选定分辨率。")
print(f"     PI 对照上方 UMAP 各 resolution 图 + 群大小分布，目测后自行决策。")
print(f"     决定后在下方 PARAMS 区设 CLUSTER_KEY 或设 OVER_CLUSTERING_RES。")

## 各分辨率综合风险标记

In [ ]:
# === 各分辨率综合风险标记（只标记，不自动选定分辨率）===
# 逐 resolution 汇总四类风险信号，帮助研究者快速定位每个分辨率的隐患：
#   ① 过度碎片化：cluster 数超预期上限，或存在小于 MIN_CLUSTER_SIZE 的微簇（统计功效弱）→ 疑似过度聚类
#   ② 样本特异簇：来自上方来源构成检测的 batch_driven cluster 计数（疑似批次残留或数据集特异群）
#   ③ marker 证据：能否为该分辨率生成 top DEG（MARKER_PREVIEW 关闭时标 "marker 证据未生成"）
#   ④ doublet/uncertain 富集：是否有 cluster 的 uncertain doublet 或 doublet_score 异常偏高
print("\n===== 各分辨率综合风险标记（仅提示，不替研究者选定）=====\n")

# 从上方来源构成结果按 resolution 统计 batch_driven cluster 数。
# 用 .get 容错：若来源构成 cell（改-2）尚未运行，此处退化为空，不报错（顺序解耦）。
_sc = adata.uns.get("05_source_composition", {})
_bd_rec = _sc.get("batch_driven_clusters", {}) if isinstance(_sc, dict) else {}
_bd_res_list = list(_bd_rec.get("resolution", [])) if isinstance(_bd_rec, dict) else []

# marker 证据是否可生成：依赖 MARKER_PREVIEW_ENABLED（本 stage 的 rank_genes_groups 预览开关）。
_marker_available = bool(MARKER_PREVIEW_ENABLED)

# doublet 信息可用性：优先看 uncertain 标签，其次看连续 doublet_score。
_has_uncertain = "doublet_prediction" in adata.obs.columns
_has_dscore = ("doublet_score" in adata.obs.columns
               and adata.obs["doublet_score"].notna().any())
_global_uncertain = None
if _has_uncertain:
    _global_uncertain = float((adata.obs["doublet_prediction"] == "uncertain").mean())

_risk = {"resolution": [], "n_clusters": [], "n_small_clusters": [],
         "over_fragmented": [], "sample_specific_count": [],
         "marker_evidence": [], "doublet_enriched": []}

for res in RESOLUTIONS:
    _rk = f"leiden_res_{res}"
    if _rk not in adata.obs.columns:
        continue
    _counts = adata.obs[_rk].value_counts()
    _n_cl = int(_counts.shape[0])
    _n_small = int((_counts < MIN_CLUSTER_SIZE).sum())
    # ① 过度碎片化：超预期上限 或 存在微簇。
    _over_frag = bool(_n_cl > EXPECTED_CLUSTER_MAX or _n_small > 0)
    # ② 样本特异簇计数（该 resolution 下 batch_driven cluster 数）。
    _ss_count = int(sum(1 for _r in _bd_res_list if float(_r) == float(res)))
    # ③ marker 证据。
    _marker_flag = "可生成" if _marker_available else "marker 证据未生成"
    # ④ doublet/uncertain 富集：某 cluster uncertain 占比 > max(2×全局, 0.3) 视为异常。
    _dbl_enriched = False
    if _has_uncertain and _global_uncertain is not None:
        _frac = adata.obs.groupby(_rk, observed=True)["doublet_prediction"].apply(
            lambda s: float((s == "uncertain").mean()))
        _thr = max(2 * _global_uncertain, 0.3)
        _dbl_enriched = bool((_frac > _thr).any())
    elif _has_dscore:
        _gm = float(adata.obs["doublet_score"].mean())
        _cm = adata.obs.groupby(_rk, observed=True)["doublet_score"].mean()
        _dbl_enriched = bool((_cm > 2 * _gm).any())

    _risk["resolution"].append(float(res))
    _risk["n_clusters"].append(_n_cl)
    _risk["n_small_clusters"].append(_n_small)
    _risk["over_fragmented"].append(_over_frag)
    _risk["sample_specific_count"].append(_ss_count)
    _risk["marker_evidence"].append(_marker_flag)
    _risk["doublet_enriched"].append(_dbl_enriched)

risk_summary_df = pd.DataFrame(_risk)
if len(risk_summary_df) > 0:
    print(risk_summary_df.to_string(index=False))
else:
    print("无可用聚类列，跳过风险标记。")
print("\n  ⚠️ 以上仅为风险提示——不替研究者选定分辨率。")
print("     疑似过度聚类 / 样本特异簇 / marker 证据未生成 / doublet 富集，"
      "均需结合上方 UMAP 与生物学先验综合判断。")

adata.uns["05_resolution_risk"] = _risk

In [ ]:
# Doublet cluster 筛查：per-cluster mean(doublet_score)
# 如果某 cluster 平均 doublet score > 2× 全局均值 → 疑似 doublet cluster
if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
    # 使用 PI 选定的分辨率（或默认最后一个 RESOLUTIONS）
    _chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
    _chosen_key = f"leiden_res_{_chosen_res}"
    if _chosen_key in adata.obs.columns:
        global_mean = adata.obs["doublet_score"].mean()
        cluster_doublet = adata.obs.groupby(_chosen_key)["doublet_score"].mean()
        suspect = cluster_doublet[cluster_doublet > 2 * global_mean]
        print(f"===== Doublet Cluster 筛查（resolution={_chosen_res}）=====")
        print(f"全局 mean doublet_score: {global_mean:.4f}")
        if len(suspect) > 0:
            print(f"⚠️ 疑似 doublet cluster（mean > 2×全局均值）:")
            for cl, score in suspect.items():
                n_cells = (adata.obs[_chosen_key] == cl).sum()
                print(f"  Cluster {cl}: mean_doublet={score:.4f} ({score/global_mean:.1f}×), n_cells={n_cells}")
            print("  → 建议在 06 注释时特别关注这些 cluster，可能需要移除")
        else:
            print("✓ 无疑似 doublet cluster")
else:
    print("跳过 doublet cluster 筛查：doublet_score 列不可用")


In [ ]:
# --- 沉默 cluster 检测：top marker 与邻近 cluster 高度重叠的 cluster 可能是噪声或过分裂 ---
if MARKER_PREVIEW_ENABLED:
    _last_res = RESOLUTIONS[-1]
    _last_key = f"leiden_res_{_last_res}"
    sc.tl.rank_genes_groups(adata, groupby=_last_key, method="wilcoxon", n_genes=5, use_raw=False)
    result = adata.uns["rank_genes_groups"]
    groups = list(result["names"].dtype.names)
    # 每个 cluster 的 top5 基因集
    cluster_markers = {}
    for g in groups:
        cluster_markers[g] = set(result["names"][g][:5])
    # 对比邻近 cluster 的 marker 重叠
    silent_pairs = []
    for i, g1 in enumerate(groups):
        for g2 in groups[i+1:]:
            jaccard = len(cluster_markers[g1] & cluster_markers[g2]) / max(len(cluster_markers[g1] | cluster_markers[g2]), 1)
            if jaccard > 0.4:
                silent_pairs.append((g1, g2, jaccard))
    if silent_pairs:
        print(f"⚠️ {len(silent_pairs)} 对 cluster 的 top-5 marker 高度重叠（Jaccard > 0.4）:")
        for g1, g2, j in sorted(silent_pairs, key=lambda x: -x[2])[:5]:
            print(f"  Cluster {g1} vs {g2}: Jaccard={j:.2f} → 考虑合并")
    else:
        print("✓ 无沉默 cluster 对（各 cluster marker 特异）")
    del adata.uns["rank_genes_groups"]


## Cluster 层级结构（Dendrogram）

Dendrogram 展示各 cluster 在嵌入空间中的层级相似性。相邻分支上的 cluster
表达谱相近，是合并的首选候选——尤其在高分辨率（过聚类）策略下，
通过 dendrogram 识别可合并的姐妹 cluster 是注释前的关键步骤。


In [ ]:
# Cluster 层级结构（dendrogram）——辅助判断哪些 cluster 可以合并
_chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
_chosen_key = f"leiden_res_{_chosen_res}"
if _chosen_key in adata.obs.columns:
    print(f"===== Dendrogram（resolution={_chosen_res}）=====")
    sc.tl.dendrogram(adata, groupby=_chosen_key, use_rep=USE_REP)
    sc.pl.dendrogram(adata, groupby=_chosen_key)
    plt.tight_layout(); plt.show()
    print("相邻分支上的 cluster 是合并的首选候选")


## QC 残留诊断

即使 01 已做 QC 过滤，仍可能有低质量细胞残留在数据中并形成独立 cluster。
本诊断检查每个 cluster 的 QC 指标分布（pct_mt、doublet_score、log_complexity），
标记中位值显著高于全局水平的 cluster——提示可能需要收紧 01 的 QC 阈值。


In [ ]:
# QC 残留诊断：检查是否有 cluster 被 QC 遗漏的低质量细胞主导
# 如果某 cluster 的中位 pct_mt / doublet_score 是 outlier → 提示回 01 收紧 QC
_chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
_chosen_key = f"leiden_res_{_chosen_res}"
if _chosen_key in adata.obs.columns:
    print(f"===== QC 残留诊断（resolution={_chosen_res}）=====")
    qc_metrics = ["pct_counts_mt"]
    if "doublet_score" in adata.obs.columns:
        qc_metrics.append("doublet_score")
    if "log_complexity" in adata.obs.columns:
        qc_metrics.append("log_complexity")

    fig, axes = plt.subplots(1, len(qc_metrics), figsize=(6*len(qc_metrics), 5))
    if len(qc_metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, qc_metrics):
        sc.pl.violin(adata, keys=metric, groupby=_chosen_key, rotation=45, ax=ax, show=False)
        ax.set_title(f"{metric} per cluster")
        # 标记 outlier cluster（中位值 > 全局中位 + 2*MAD）
        global_med = adata.obs[metric].median()
        global_mad = median_abs_deviation(adata.obs[metric].dropna(), nan_policy="omit")
        # 用颜色标记 outlier——scanpy violin plot 的 patch collection 索引对应 sorted unique groups
        clusters_sorted = sorted(adata.obs[_chosen_key].unique())
        for i, cl in enumerate(clusters_sorted):
            cl_med = adata.obs.loc[adata.obs[_chosen_key] == cl, metric].median()
            if cl_med > global_med + 2 * global_mad:
                # 尝试修改 violin patch 颜色（兼容 scanpy >= 1.10）
                for collection in ax.collections:
                    if hasattr(collection, "set_facecolor"):
                        try:
                            face_colors = collection.get_facecolors()
                            if len(face_colors) > i:
                                face_colors[i] = [1.0, 0.0, 0.0, 0.6]
                                collection.set_facecolors(face_colors)
                        except (IndexError, ValueError):
                            pass
    plt.tight_layout(); plt.show()

    # 文字摘要
    for metric in qc_metrics:
        global_med = adata.obs[metric].median()
        global_mad = median_abs_deviation(adata.obs[metric].dropna(), nan_policy="omit")
        outlier_clusters = []
        for cl in sorted(adata.obs[_chosen_key].unique()):
            cl_med = adata.obs.loc[adata.obs[_chosen_key] == cl, metric].median()
            if cl_med > global_med + 2 * global_mad:
                outlier_clusters.append(f"{cl}(med={cl_med:.2f})")
        if outlier_clusters:
            print(f"⚠️ {metric} outlier clusters: {', '.join(outlier_clusters)}")
            print(f"  全局 median={global_med:.2f}, threshold={global_med + 2*global_mad:.2f}")
        else:
            print(f"✓ {metric}: 无 outlier cluster")


## 跨数据集 cluster 归属偏差

即使 04 做了 batch correction，仍可能有某些 cluster 被单一数据集主导。
这个诊断检查每个 cluster 的细胞来源分布：

- **>80% 来自单一数据集**：需判断是真实的生物学差异（某数据集含独特组织区域）
  还是 batch correction 不完全（该 cluster 本质上是数据集批次效应）
- **分布均衡**：说明 batch correction 在此分辨率下有效，跨数据集的细胞类型具有可比性

如果发现严重偏差：回到 04 检查 UMAP batch 着色，判断是否需要调整校正参数。

In [ ]:
# === 跨数据集 cluster 归属偏差检测 ===
# 逐 resolution × 分层键计算每个 cluster 的来源构成，标记被单一来源主导的 cluster。
# 为什么逐 resolution：cluster 编号随分辨率变化，只看单一中间分辨率会漏掉高/低分辨率下才
# 出现的偏差群。偏差既可能是真实数据集特异群，也可能是 04 批次校正不完全——二者处理相反，
# 需回 04 UMAP batch 着色人工判断，本 cell 只负责标记。

# batch 主键检测：优先读 04 写入的 batch_key，再回退到常见来源列名。
_batch_col = None
if "harmony_v1" in adata.uns and "batch_key" in adata.uns["harmony_v1"]:
    _batch_col = adata.uns["harmony_v1"]["batch_key"]
elif "scvi_v1" in adata.uns and "batch_key" in adata.uns["scvi_v1"]:
    _batch_col = adata.uns["scvi_v1"]["batch_key"]
else:
    for cand in ["source_dataset", "batch", "sample_id"]:
        if cand in adata.obs.columns:
            _batch_col = cand
            break

# 实际可用的分层维度：COMPOSITION_KEYS 中真正存在于 obs 的列。
_avail_comp_keys = [k for k in COMPOSITION_KEYS if k in adata.obs.columns]

# batch-driven 记录用 columnar dict（等长列表的列式结构）存储：
# anndata 无法把 list-of-dict 写进 h5ad（下游 draft 会 write adata），列式结构可安全
# round-trip，且下游 pd.DataFrame(...) 一行还原。
_bd = {"resolution": [], "cluster": [], "dominant_source": [],
       "dominant_pct": [], "n_cells": [], "flag": []}

if not _avail_comp_keys:
    print("未找到任何来源列（COMPOSITION_KEYS 均不在 obs），跳过来源构成检测（写空表，不报错）。")
else:
    print(f"===== 逐分辨率跨数据集来源构成（阈值={BATCH_DOMINANCE_THRESHOLD:.0%}）=====")
    print(f"分层维度: {_avail_comp_keys}；batch 主键: {_batch_col}\n")

    # batch 主键：优先用检测到的 _batch_col（若它在可用分层键内），否则退第一个可用键。
    _primary_key = _batch_col if (_batch_col in _avail_comp_keys) else _avail_comp_keys[0]

    for res in RESOLUTIONS:
        _res_key = f"leiden_res_{res}"
        if _res_key not in adata.obs.columns:
            continue
        # 逐分层键计算每 cluster 各来源占比（normalize="index" = 每行按 cluster 归一化）。
        for _ck in _avail_comp_keys:
            _ct = pd.crosstab(adata.obs[_res_key], adata.obs[_ck], normalize="index")
            # 只在 batch 主键上做主导判定；其余分层键的 crosstab 仅用于展示，不重复标记。
            if _ck == _primary_key:
                _dominant_pct = _ct.max(axis=1)
                _biased = _dominant_pct[_dominant_pct > BATCH_DOMINANCE_THRESHOLD]
                for cl in _biased.index:
                    _dom = _ct.loc[cl].idxmax()
                    _bd["resolution"].append(float(res))
                    _bd["cluster"].append(str(cl))
                    _bd["dominant_source"].append(str(_dom))
                    _bd["dominant_pct"].append(float(_dominant_pct[cl]))
                    _bd["n_cells"].append(int((adata.obs[_res_key] == cl).sum()))
                    _bd["flag"].append("batch_driven")

    _n_flagged = len(_bd["resolution"])
    if _n_flagged > 0:
        print(f"⚠️ 共标记 {_n_flagged} 个疑似 batch-driven cluster"
              f"（单一来源占比 > {BATCH_DOMINANCE_THRESHOLD:.0%}，按 {_primary_key} 判定）:")
        for _i in range(_n_flagged):
            print(f"    res={_bd['resolution'][_i]} cluster {_bd['cluster'][_i]} "
                  f"({_bd['n_cells'][_i]} cells): {_bd['dominant_pct'][_i]:.0%} "
                  f"来自 {_bd['dominant_source'][_i]}")
        print("  → 判断依据：真实数据集特异群（保留）vs 批次校正不完全（回 04 检查 UMAP batch 着色）。")
    else:
        print(f"✓ 所有分辨率下各 cluster 跨来源分布均衡（无 > {BATCH_DOMINANCE_THRESHOLD:.0%} 单源主导）。")

# 写回 uns：threshold + 使用的分层键 + batch-driven 记录（columnar，h5ad 安全）。
adata.uns["05_source_composition"] = {
    "threshold": float(BATCH_DOMINANCE_THRESHOLD),
    "keys_used": _avail_comp_keys,
    "batch_driven_clusters": _bd,
}

## Sanity Marker 验证

用已知生物学 marker 验证嵌入空间的基本分离格局是否正确。这不是正式注释，
只是快速 sanity check——如果 EPCAM（上皮）、PTPRC/CD45（免疫）、VIM（间质）
三大 compartment 在 UMAP 上没有明显分离，说明上游嵌入或 HVG 选择有问题。


In [ ]:
# --- Cluster 纯度：当有先验注释时，评估每个 cluster 的细胞类型一致性 ---
_annot_cols = [c for c in adata.obs.columns if c.startswith("cell_type_original_")]
if _annot_cols:
    _annot_col = _annot_cols[0]
    _valid = adata.obs[_annot_col].notna()
    if _valid.sum() > 50:
        _chosen_res = OVER_CLUSTERING_RES if OVER_CLUSTERING_RES else RESOLUTIONS[-1]
        _chosen_key = f"leiden_res_{_chosen_res}"
        if _chosen_key in adata.obs.columns:
            print(f"===== Cluster 纯度（基于 {_annot_col}）=====")
            purity_data = []
            for cl in sorted(adata.obs[_chosen_key].unique()):
                mask = (adata.obs[_chosen_key] == cl) & _valid
                if mask.sum() == 0:
                    continue
                counts = adata.obs.loc[mask, _annot_col].value_counts()
                top_type = counts.index[0]
                purity = counts.iloc[0] / mask.sum()
                purity_data.append({"cluster": cl, "dominant_type": top_type,
                                    "purity": purity, "n_cells": mask.sum()})
            purity_df = pd.DataFrame(purity_data)
            low_purity = purity_df[purity_df["purity"] < 0.6]
            if len(low_purity) > 0:
                print(f"⚠️ {len(low_purity)} 个 cluster 纯度 < 60%（混合 cluster）:")
                print(low_purity.to_string(index=False))
            else:
                print(f"✓ 全部 cluster 纯度 ≥ 60%")
            print(f"\n平均纯度: {purity_df['purity'].mean():.1%}")


In [ ]:
# Sanity Marker 验证：已知生物学分离格局是否正确？
# 如果 EPCAM+/PTPRC+/VIM+ 三大 compartment 没有分开 → 上游有问题
if SANITY_MARKERS:
    available_markers = {name: gene for name, gene in SANITY_MARKERS.items() if gene in adata.var_names}
    missing = {name: gene for name, gene in SANITY_MARKERS.items() if gene not in adata.var_names}
    if missing:
        print(f"⚠️ 部分 sanity marker 不在数据中: {missing}")
    if available_markers:
        print(f"===== Sanity Marker 验证 =====")
        genes_to_plot = list(available_markers.values())
        sc.pl.umap(adata, color=genes_to_plot, ncols=min(len(genes_to_plot), 4),
                   use_raw=False, show=True)
        print("检查：各 marker 是否形成明显分离的区域？")
        print("  EPCAM+ = 上皮 | PTPRC(CD45)+ = 免疫 | VIM+ = 间质 | MKI67+ = 增殖")
        print("  如果全部混在一起 → 可能 04 过校正或 03 HVG 有问题")


In [ ]:
# === 胃粘膜特异性 marker UMAP ===
# 为什么在 SANITY_MARKERS 之外单独画胃粘膜标记？
# SANITY_MARKERS（EPCAM/PTPRC/VIM/MKI67）只能判断四大 compartment 是否分开，
# 但无法回答"上皮内部有没有分出壁细胞、主细胞、黏液细胞、肠化"。
# 胃粘膜上皮亚型的分辨是胃癌前病变分析的核心——如果聚类阶段就没捕获这些亚型，
# 下游 SPEM/IM 注释和分析将失去基础。
# 注意：这个 UMAP 与上方 sanity marker UMAP 共用 adata 中的 X_umap（如有），
# 不需要重新计算 UMAP 坐标。
if GASTRIC_MARKERS:
    _gastric_avail = {name: gene for name, gene in GASTRIC_MARKERS.items()
                      if gene in adata.var_names}
    if _gastric_avail:
        print(f"\n--- 胃粘膜谱系 marker UMAP ({len(_gastric_avail)}/{len(GASTRIC_MARKERS)}) ---")
        _missing_gastric = {n: g for n, g in GASTRIC_MARKERS.items() if g not in adata.var_names}
        if _missing_gastric:
            print(f"  未找到: {_missing_gastric}")

        sc.pl.umap(adata, color=list(_gastric_avail.values()),
                   ncols=4, frameon=False, show=True,
                   save="_05_gastric_markers.png")

    else:
        print("胃粘膜 marker 均不在 var_names 中，跳过")


## Compartment 归属诊断与分层聚类建议

**为什么单一 resolution 不够？**

上皮 compartment 内部存在连续的 SPEM/IM 渐变——需要高 resolution（1.0-2.0）
才能分离出壁细胞、主细胞、颈黏液细胞、表面黏液细胞、SPEM、IM、异型增生的亚型梯度。
但在同样高 resolution 下，免疫 compartment 会过度碎片化（T/B/Myeloid 亚型被拆成数十个碎片），
增加下游注释负担且统计功效下降。

**分层聚类（hierarchical clustering）**是资深专家的标准做法：
1. 先在中等分辨率（0.6-1.0）粗分 compartment（上皮/免疫/间质/增殖）
2. 再对感兴趣的 compartment（通常是上皮）取子集（subset），用更高分辨率重聚类
3. 各 compartment 独立注释后再合并结果

下方细胞自动识别各 cluster 的主导 compartment 并给出 subset 重聚类建议。

In [ ]:
# === Compartment 归属诊断 + 分层聚类建议 ===
# 单一 resolution 不能同时满足所有 compartment——上皮内部的 SPEM/IM 渐变需要
# 高 resolution 才能分离，而免疫 compartment 在同样高 res 下会过度碎片化。
# 分层聚类（先粗分 compartment → 再对 subset 重聚类）是资深专家的标准做法。
# 本 cell 用 sanity marker 的表达模式帮助判断哪个 compartment 最需要 subset 重聚类。
if SANITY_MARKERS:
    # 选择一个中等分辨率做诊断（RESOLUTIONS 中间偏高位置）
    _diag_res = RESOLUTIONS[len(RESOLUTIONS) // 2]  # 约 0.8 或 1.0
    _diag_key = f"leiden_res_{_diag_res}"

    if _diag_key in adata.obs.columns:
        print(f"===== Compartment 归属诊断（基于 {_diag_key}）=====\n")

        # 计算每个 cluster 的各 compartment marker 平均表达
        _available = {name: gene for name, gene in SANITY_MARKERS.items()
                      if gene in adata.var_names}

        if _available:
            _profiles = []
            for cl in sorted(adata.obs[_diag_key].unique(), key=lambda x: int(x)):
                _mask = adata.obs[_diag_key] == cl
                _row = {"cluster": cl, "n_cells": int(_mask.sum())}
                for name, gene in _available.items():
                    _expr = adata[_mask, gene].X
                    if sp.issparse(_expr):
                        _expr = _expr.toarray()
                    _row[name] = float(_expr.mean())
                _profiles.append(_row)

            _prof_df = pd.DataFrame(_profiles)

            # 判断每个 cluster 的主导 compartment（marker 平均表达最高者）
            _marker_cols = [c for c in _prof_df.columns if c not in ["cluster", "n_cells"]]
            _prof_df["dominant"] = _prof_df[_marker_cols].idxmax(axis=1)

            # 分组输出
            _compartments = {}
            for _, row in _prof_df.iterrows():
                comp = row["dominant"]
                if comp not in _compartments:
                    _compartments[comp] = []
                _compartments[comp].append(str(row["cluster"]))

            for comp, clusters in sorted(_compartments.items()):
                print(f"  {comp} 为主: clusters {', '.join(clusters)}")

            # 检测混合 compartment cluster（多个 marker 都高表达 → 可能欠分）
            print()
            _mixed = []
            for _, row in _prof_df.iterrows():
                _vals = row[_marker_cols]
                _high = _vals[_vals > _vals.median() + _vals.std()].index.tolist()
                if len(_high) >= 2:
                    _mixed.append((row["cluster"], _high))

            if _mixed:
                print("  ⚠️ 混合 compartment cluster（可能欠分）:")
                for cl, comps in _mixed:
                    print(f"    Cluster {cl}: {' + '.join(comps)} 均高表达")
            else:
                print("  ✓ 无明显混合 compartment cluster")

            # --- 胃粘膜标记辅助证据：对上皮 dominant 的 cluster 检查胃亚型 marker 表达 ---
            if GASTRIC_MARKERS and "Epithelial" in _compartments:
                _gastric_avail_diag = {name: gene for name, gene in GASTRIC_MARKERS.items()
                                       if gene in adata.var_names}
                if _gastric_avail_diag:
                    print(f"\n--- 胃粘膜亚型标记辅助证据（上皮 compartment）---")
                    _epi_clusters = _compartments["Epithelial"]
                    for cl in _epi_clusters:
                        _mask = adata.obs[_diag_key] == cl
                        _gastric_scores = {}
                        for name, gene in _gastric_avail_diag.items():
                            _expr = adata[_mask, gene].X
                            if sp.issparse(_expr):
                                _expr = _expr.toarray()
                            _gastric_scores[name] = float(_expr.mean())
                        # 找出最高表达的 3 个胃亚型 marker
                        _top3 = sorted(_gastric_scores.items(), key=lambda x: -x[1])[:3]
                        _top_str = ", ".join(f"{n}={v:.2f}" for n, v in _top3)
                        print(f"  Cluster {cl}: top gastric markers → {_top_str}")
                    print("  → 如果上皮 cluster 缺乏明确的胃亚型 marker（全部 <0.5），")
                    print("    可能说明该 cluster 是低质量/非上皮污染，或聚类粒度不够")

            # 分层聚类建议
            print(f"\n--- 分层聚类建议 ---")
            if "Epithelial" in _compartments:
                _epi_clusters = _compartments["Epithelial"]
                _epi_n = _prof_df[_prof_df["cluster"].isin(_epi_clusters)]["n_cells"].sum()
                print(f"  上皮 subset (clusters {', '.join(_epi_clusters)}, ~{_epi_n:,} cells):")
                print(f"    → 建议在 stage 06c 中单独取子集重聚类（res=1.0-2.0）")
                print(f"    → 目标：分离壁细胞/主细胞/颈黏液/表面黏液/SPEM/IM/异型增生")
            if "Immune" in _compartments:
                _imm_clusters = _compartments["Immune"]
                _imm_n = _prof_df[_prof_df["cluster"].isin(_imm_clusters)]["n_cells"].sum()
                print(f"  免疫 subset (clusters {', '.join(_imm_clusters)}, ~{_imm_n:,} cells):")
                print(f"    → 如需细分 T/B/Myeloid 亚型，同样建议子集重聚类")
                print(f"    → 如只需大类注释（T/B/Myeloid/NK），当前分辨率可能够用")
            if "Stromal" in _compartments:
                _stro_clusters = _compartments["Stromal"]
                _stro_n = _prof_df[_prof_df["cluster"].isin(_stro_clusters)]["n_cells"].sum()
                print(f"  间质 subset (clusters {', '.join(_stro_clusters)}, ~{_stro_n:,} cells):")
                print(f"    → 如需分离成纤维/平滑肌/周细胞/间皮亚型，建议子集重聚类")

## 运行元数据

版控键记录聚类参数与追溯链，供后续审计查询。

## 研究者决策：显式选择聚类分辨率\n\n对照上方各 resolution 的 UMAP、群大小分布、跨数据集来源构成表与综合风险表，综合生物学先验判断后，在 PARAMS 区设定 `SELECTED_CLUSTER_KEY`（形如 `"leiden_res_0.6"`）。系统只给机器建议（`recommended_resolution`），不自动替研究者选定；未选定时本 stage 保持 NEEDS_REVIEW。

In [ ]:
# === 研究者显式选择 selected_cluster_key（不自动选）===
# 对照上方 UMAP / 群大小 / 来源构成 / 综合风险表判断后，在 PARAMS 区设 SELECTED_CLUSTER_KEY。
# 本 cell 绝不根据指标自动赋值——机器只给建议（recommended_resolution），最终选定由研究者负责。
if SELECTED_CLUSTER_KEY is None:
    # fresh-kernel Run-All / CI 无人值守场景：未选择时不 raise，保持 NEEDS_REVIEW。
    print("尚未选择聚类列（SELECTED_CLUSTER_KEY=None）。")
    print("Stage 05 保持 NEEDS_REVIEW，等待研究者在 PARAMS 区显式设定 "
          "SELECTED_CLUSTER_KEY 后重跑。")
else:
    # 显式选择校验：类型 + 存在性 + resolution 在 sweep 范围内 + 真的多于一个 cluster。
    if not isinstance(SELECTED_CLUSTER_KEY, str) or SELECTED_CLUSTER_KEY not in adata.obs.columns:
        raise ValueError(
            f"SELECTED_CLUSTER_KEY={SELECTED_CLUSTER_KEY!r} not in obs columns；"
            f"请填入实际存在的聚类列名（形如 'leiden_res_0.6'）。"
        )
    _m = re.fullmatch(r"leiden_res_(.+)", SELECTED_CLUSTER_KEY)
    if _m is None:
        raise ValueError(
            f"SELECTED_CLUSTER_KEY={SELECTED_CLUSTER_KEY!r} 不符合 'leiden_res_{{r}}' 命名，"
            f"无法解析 resolution。"
        )
    _sel_res = float(_m.group(1))
    if _sel_res not in [float(r) for r in RESOLUTIONS]:
        raise ValueError(
            f"selected resolution {_sel_res} not swept（不在 RESOLUTIONS={RESOLUTIONS} 中）。"
        )
    if adata.obs[SELECTED_CLUSTER_KEY].nunique() < 2:
        raise ValueError(
            f"SELECTED_CLUSTER_KEY={SELECTED_CLUSTER_KEY!r} 只有单一 cluster，无法用于下游注释。"
        )

    adata.uns["selected_cluster_key"] = SELECTED_CLUSTER_KEY
    adata.uns["selection_rationale"] = SELECTION_RATIONALE
    _n_sel = int(adata.obs[SELECTED_CLUSTER_KEY].nunique())
    print(f"✓ 已选定聚类列: {SELECTED_CLUSTER_KEY}（{_n_sel} clusters）")
    print(f"  选择依据: {SELECTION_RATIONALE or '（未填写 SELECTION_RATIONALE）'}")
    # 附带该分辨率的来源构成风险摘要（从 05_source_composition 提取，.get 容错）。
    _sc2 = adata.uns.get("05_source_composition", {})
    _bd2 = _sc2.get("batch_driven_clusters", {}) if isinstance(_sc2, dict) else {}
    _bd2_res = list(_bd2.get("resolution", [])) if isinstance(_bd2, dict) else []
    _n_bd = sum(1 for _r in _bd2_res if float(_r) == _sel_res)
    if _n_bd > 0:
        print(f"  ⚠️ 该分辨率下有 {_n_bd} 个疑似 batch-driven cluster，"
              f"注释时需留意（详见 uns['05_source_composition']）。")
    else:
        print("  该分辨率下无疑似 batch-driven cluster。")

In [ ]:
# === 推荐分辨率（基于扫描指标自动推荐，供 stage 06 PARAMS 参考）===
# 当有 silhouette 指标时选最高分；无指标时选中间分辨率。
# 这是启发式推荐——最终决策以 UMAP 目测为准。
_rec_res = None
if _sweep_completed:
    _df_metrics = pd.DataFrame(results)
    if "silhouette" in _df_metrics.columns:
        _rec_idx = _df_metrics["silhouette"].idxmax()
        _rec_res = _df_metrics.loc[_rec_idx, "resolution"]
    elif len(_df_metrics) > 0:
        _rec_res = RESOLUTIONS[len(RESOLUTIONS) // 2]
elif len(RESOLUTIONS) > 0:
    _rec_res = RESOLUTIONS[len(RESOLUTIONS) // 2]

if _rec_res is not None:
    adata.uns["recommended_resolution"] = _rec_res
    print(f"\n推荐分辨率: {_rec_res}（基于聚类质量指标）")
    print(f'  → stage 06 设置: LEIDEN_COL = "leiden_res_{_rec_res}"')

# 记录聚类运行元数据——包含参数、版本与追溯链。
print("\n===== Run metadata =====")

adata.uns["leiden_v1"] = {
    "use_rep": USE_REP,
    "resolutions": RESOLUTIONS,
    "flavor": "igraph",
    "recommended_resolution": _rec_res,
    "n_neighbors": adata.uns["neighbors"]["params"]["n_neighbors"],
    "timestamp": datetime.datetime.now().isoformat(),
}

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "05_clustered"     # 本 stage 标识
adata.uns["run_id"] = RUN_ID
adata.uns["upstream"] = [str(UPSTREAM_CHECKPOINT)]
adata.uns["upstream_inputs"] = {"stage04": upstream_input}
adata.uns["status"] = "NEEDS_REVIEW"

# 各分辨率的簇数汇总
cluster_summary = {}
for col in leiden_cols:
    cluster_summary[col] = int(adata.obs[col].nunique())
adata.uns["leiden_v1"]["cluster_counts"] = cluster_summary

print("Clusters per resolution:")
for col, n in cluster_summary.items():
    print(f"  {col}: {n}")
print(f"status: {adata.uns['status']}")

In [ ]:
# 写入前自检——确保 X 保持稀疏 float32，防止意外 densify 导致内存暴涨。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 意外退化: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("自检通过: X 为稀疏 CSR float32。")

In [ ]:
# === Stage 05 draft checkpoint：等待研究者选择 ===
candidate_cluster_keys = [f"leiden_res_{res}" for res in RESOLUTIONS]
present_keys = [key for key in candidate_cluster_keys if key in adata.obs.columns]
complete_keys = [key for key in present_keys if bool(adata.obs[key].notna().all())]
multi_cluster_keys = [key for key in complete_keys if adata.obs[key].nunique() >= 2]
output_filename_valid = (
    isinstance(OUTPUT_FILENAME, str) and bool(OUTPUT_FILENAME.strip())
    and OUTPUT_FILENAME not in {".", "..", "manifest.json"}
    and not Path(OUTPUT_FILENAME).is_absolute()
    and Path(OUTPUT_FILENAME).name == OUTPUT_FILENAME
    and "/" not in OUTPUT_FILENAME and "\\" not in OUTPUT_FILENAME
    and re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*\.h5ad", OUTPUT_FILENAME) is not None
    and not any(ord(char) < 32 or ord(char) == 127 for char in OUTPUT_FILENAME)
)
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "candidate_resolutions_valid": bool(candidate_cluster_keys) and len(set(candidate_cluster_keys)) == len(candidate_cluster_keys),
    "candidate_columns_present": present_keys == candidate_cluster_keys,
    "candidate_columns_complete": complete_keys == candidate_cluster_keys,
    "candidate_columns_have_multiple_clusters": multi_cluster_keys == candidate_cluster_keys,
    "output_filename_valid": output_filename_valid,
}
stage_status = determine_stage_status(
    {}, hard_postconditions, needs_review=True, allow_no_required_methods=True
)
effective_parameters = snapshot_effective_parameters(
    globals(), exclude=("UPSTREAM_CHECKPOINT",), path_root=Path(_root)
)
runtime_provenance = collect_runtime_provenance(
    _root, ("anndata", "scanpy", "numpy", "pandas", "scipy", "scikit-learn", "igraph", "leidenalg")
)
manifest_payload = {
    "run_id": RUN_ID, "stage": "05_clustered", "stage_status": stage_status.value,
    "inputs": [upstream_input], "effective_parameters": effective_parameters,
    "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions,
    "candidate_cluster_keys": candidate_cluster_keys,
    "cluster_counts": {key: int(adata.obs[key].nunique()) for key in present_keys},
    "selected_cluster_key": SELECTED_CLUSTER_KEY,
    "selection_rationale": SELECTION_RATIONALE,
    "source_composition_summary": adata.uns.get("05_source_composition", {}).get("batch_driven_clusters", []),
}
run_paths = prepare_run(RUN_ROOT, RUN_ID)
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 05 FAILED: {hard_postconditions}")

adata.uns["stage"] = "05_clustered"
adata.uns["status"] = stage_status.value
adata.uns["run_id"] = RUN_ID
adata.uns["upstream"] = [str(UPSTREAM_CHECKPOINT)]
adata.uns["upstream_inputs"] = {"stage04": upstream_input}
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
draft_output_path = validate_checkpoint(run_paths.manifest_path)
print(f"已保存待审查 draft: {draft_output_path}")
print("stage_status: NEEDS_REVIEW；未选择分群列，未创建 promoted 或旧式兼容路径。")

### Stage 05 Verdict

本 stage 当前固定为 `NEEDS_REVIEW`，draft 不会自动进入 06。后续 PR5 提供研究者选择与安全提升流程：
- [ ] 选定分辨率（LEIDEN_COL）-> 设到 06
- [ ] 无单数据集主导的 cluster（或已确认为真实差异）
- [ ] Compartment 归属明确（上皮/免疫/间质各有对应 cluster）


In [ ]:
# 跨 stage 边界释放内存。
# 如果 Jupyter 内核会话中接着跑下一 stage（06），
# 这一步避免两个 stage 的 AnnData 同时驻留内存导致 OOM。
del adata
import gc
gc.collect()
print("内存已释放。")

## 🗂 UX-3 Run 管理与跨参数比较

> 此区块只读：枚举当前 `RUN_ROOT` 下所有 run 的状态、参数差异和清理候选。  
> **绝不自动删除或提升**——动作（pin/删除）须手动在文件系统执行。  
> `pinned.marker` 文件存在于 run 目录下 → 该 run 被保留（PINNED）。


In [ ]:
# === UX-3 wave2 | run 状态总览 ===
# enumerate_run_manifests 只读 json + os.stat，不载入 h5ad 矩阵（内存安全）
from pathlib import Path
import pandas as pd
from scrna_integration.run_contract import (
    enumerate_run_manifests,
    diff_effective_parameters,
    enumerate_cleanup_candidates,
)

try:
    from IPython.display import display as _display
except ImportError:
    _display = print  # 非 Jupyter 环境退化为 print

_run_root = Path(RUN_ROOT)
if not _run_root.exists():
    print(f"⚠️ RUN_ROOT '{_run_root}' 尚不存在，暂无 run 记录。")
    _records = []
else:
    _records = enumerate_run_manifests(_run_root)

if _records:
    _rows = [
        {
            "run_id": r.run_id,
            "state": r.state,
            "category": r.category.name if r.category is not None else "—",
            "pinned": "📌" if r.pinned else "",
            "stage_status": r.stage_status,
            "output_size_mb": (
                f"{r.output_size_bytes / 1e6:.1f}" if r.output_size_bytes is not None else "—"
            ),
            "error": r.error or "",
        }
        for r in _records
    ]
    _display(pd.DataFrame(_rows).to_string(index=False))
    print(f"\n共 {len(_records)} 个 run，其中 pinned: {sum(r.pinned for r in _records)} 个")
else:
    print("ℹ️ 未找到任何 run 记录。")


In [ ]:
# === UX-3 wave2 | 跨 run 参数比较 ===
# diff_effective_parameters 仅展示有差异的参数键，帮助判断哪些 run 用了不同超参
_params_dict = {
    r.run_id: r.effective_parameters
    for r in _records
    if r.effective_parameters is not None
}

if len(_params_dict) < 2:
    print("ℹ️ 有效参数记录少于 2 个，无法比较（需要先成功跑多个 run）。")
else:
    _diff = diff_effective_parameters(_params_dict)
    _differing = _diff.get("differing_keys", [])
    if not _differing:
        print("✅ 所有 run 的有效参数完全一致。")
    else:
        print(f"⚠️ 存在差异的参数（{len(_differing)} 个）：")
        _diff_rows = []
        for _key in _differing:
            _row = {"参数": _key}
            for _rid, _info in _diff["parameters"][_key]["values"].items():
                _row[_rid] = _info["value"] if _info["present"] else "（未设置）"
            _diff_rows.append(_row)
        _display(pd.DataFrame(_diff_rows).to_string(index=False))
    print(f"\n共同参数（全 run 一致）：{len(_diff.get('shared_keys', []))} 个")


In [ ]:
# === UX-3 wave2 | 清理候选枚举 ===
# 只列出 SUPERSEDED/FAILED run 中的大文件（默认 *.h5ad）
# ⚠️ 此处仅枚举，不执行任何删除——请人工确认后手动清理
_candidates = enumerate_cleanup_candidates(_records)

if not _candidates:
    print("✅ 无清理候选（SUPERSEDED/FAILED run 中无大型 .h5ad 文件）。")
else:
    _clean_rows = [
        {
            "run_id": c.run_id,
            "category": c.category.name,
            "path": str(c.path.name),
            "size_mb": f"{c.size_bytes / 1e6:.1f}",
            "state": c.state,
        }
        for c in _candidates
    ]
    _display(pd.DataFrame(_clean_rows).to_string(index=False))
    _total_mb = sum(c.size_bytes for c in _candidates) / 1e6
    print(f"\n⚠️ 共 {len(_candidates)} 个清理候选，总大小约 {_total_mb:.1f} MB")
    print("删除命令示例（人工确认后执行）：")
    for c in _candidates[:3]:
        print(f"  rm '{c.path}'")
